# 🏗️ OpenPCDet Finetuning Pipeline
### Custom 3D Object Detection: `SilverPlate` · `SafetyCone` · `Barricade`

**Based on:** [open-mmlab/OpenPCDet](https://github.com/open-mmlab/OpenPCDet)

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1️⃣ | Environment setup (CUDA, spconv, OpenPCDet) |
| 2️⃣ | Dataset directory layout |
| 3️⃣ | Point cloud ingestion (`.bin` / `.pcd` / `.ply` → `.npy`) |
| 4️⃣ | Label file helpers (KITTI-style `.txt`) |
| 5️⃣ | Train / val / test split |
| 6️⃣ | Dataset & model YAML config generation |
| 7️⃣ | Info-file + GT-database creation |
| 8️⃣ | Finetuning with pretrained weights |
| 9️⃣ | Evaluation (mAP) |
| 🔟 | Inference on new point clouds |
| ➕ | Semi-automatic labelling helper (DBSCAN + OBB) |

> **Kaggle GPU:** This notebook is configured for a **P100 / T4** GPU (16 GB VRAM).
> Enable it via *Settings → Accelerator → GPU*.

---
## Step 1 — Environment Setup
Install `spconv` (sparse convolution back-end) and clone + build OpenPCDet.

> ⚠️ **First run takes ~5 min.** Subsequent runs are faster if you freeze the cell.

In [ ]:
import subprocess, sys

def run(cmd, **kw):
    """Helper: run a shell command and stream output."""
    print(f"▶ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)

# ── Detect CUDA version ──────────────────────────────────────────────────────
import torch
cuda_ver = torch.version.cuda          # e.g. '11.8'
cuda_tag = cuda_ver.replace('.', '')   # e.g. '118'
print(f"PyTorch {torch.__version__}  |  CUDA {cuda_ver}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── Install spconv ────────────────────────────────────────────────────────────
# Choose the wheel that matches your CUDA version (cu113 / cu116 / cu117 / cu118)
SPCONV_PKG = f"spconv-cu{cuda_tag}"
run(f"{sys.executable} -m pip install -q {SPCONV_PKG}")

# ── Extra dependencies ────────────────────────────────────────────────────────
run(f"{sys.executable} -m pip install -q open3d scikit-learn easydict pyyaml "
    "llvmlite numba SharedArray pyquaternion scikit-image tqdm")

# ── Clone OpenPCDet ───────────────────────────────────────────────────────────
import os
OPENPCDET_ROOT = "/kaggle/working/OpenPCDet"
if not os.path.exists(OPENPCDET_ROOT):
    run("git clone --depth 1 https://github.com/open-mmlab/OpenPCDet.git /kaggle/working/OpenPCDet")

# ── Build OpenPCDet (C++ ops) ─────────────────────────────────────────────────
run(f"{sys.executable} -m pip install -q -r /kaggle/working/OpenPCDet/requirements.txt")
run(f"{sys.executable} setup.py develop",
    cwd=OPENPCDET_ROOT)

print("\n✅ Environment ready.")

---
## Step 2 — Imports & Global Constants

In [ ]:
import os, sys, glob, random, shutil, pickle, logging, argparse, subprocess
from pathlib import Path
import numpy as np
import yaml

# ── Add OpenPCDet to path ─────────────────────────────────────────────────────
OPENPCDET_ROOT = Path("/kaggle/working/OpenPCDet")
sys.path.insert(0, str(OPENPCDET_ROOT))

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("/kaggle/working/data/excavation")
OUTPUT_DIR  = Path("/kaggle/working/output/excavation")
DATASET_YAML = OPENPCDET_ROOT / "tools/cfgs/dataset_configs/excavation_dataset.yaml"
MODEL_YAML   = OPENPCDET_ROOT / "tools/cfgs/custom_models/excavation_pointpillars.yaml"

# ── Class definitions ─────────────────────────────────────────────────────────
CLASS_NAMES = ["SilverPlate", "SafetyCone", "Barricade"]

MAP_CLASS_TO_KITTI = {
    "SilverPlate": "Car",        # flat/compact  → Car eval slot
    "SafetyCone":  "Pedestrian", # vertical/slim → Pedestrian eval slot
    "Barricade":   "Cyclist",    # elongated     → Cyclist eval slot
}

# Anchor sizes [l, w, h] in metres — tuned for excavation-site objects
ANCHOR_SIZES = {
    "SilverPlate": [0.60, 0.60, 0.05],  # thin flat plate
    "SafetyCone":  [0.45, 0.45, 1.00],  # standard road cone
    "Barricade":   [2.00, 0.50, 1.00],  # road barricade
}

# IoU thresholds: (matched, unmatched) per class
IOU_THRESHOLDS = {
    "SilverPlate": (0.35, 0.20),
    "SafetyCone":  (0.35, 0.20),
    "Barricade":   (0.45, 0.30),
}

POINT_CLOUD_RANGE = [-50.0, -50.0, -3.0, 50.0, 50.0, 3.0]  # [xmin,ymin,zmin,xmax,ymax,zmax]
VOXEL_SIZE        = [0.05, 0.05, 0.10]                       # small voxels → fine detail
TRAIN_RATIO, VAL_RATIO = 0.75, 0.15

print("✅ Constants loaded.")

---
## Step 3 — Dataset Directory Layout

```
data/excavation/
├── points/          ← *.npy  (N,4) float32 [x, y, z, intensity]
├── labels/          ← *.txt  one box per line: cx cy cz l w h yaw ClassName
├── ImageSets/
│   ├── train.txt
│   ├── val.txt
│   └── test.txt
└── gt_database/     ← created automatically in Step 7
```

In [ ]:
def create_dataset_layout(root: Path) -> None:
    """Create the OpenPCDet CustomDataset directory structure."""
    for d in ["points", "labels", "ImageSets", "gt_database"]:
        (root / d).mkdir(parents=True, exist_ok=True)
    log.info("Dataset layout created under: %s", root)

create_dataset_layout(DATA_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Directories created.")

---
## Step 4 — Synthetic Demo Data
Generates realistic dummy point clouds + labels so every downstream step
can be validated end-to-end **without a real LiDAR dataset**.

> 🔁 **Replace this cell** with your own data ingestion once you have real scans.

In [ ]:
# ── Point-cloud ingestion helpers ─────────────────────────────────────────────

def load_bin(filepath: str) -> np.ndarray:
    """Velodyne .bin → (N,4) float32 [x,y,z,intensity]."""
    return np.fromfile(filepath, dtype=np.float32).reshape(-1, 4)


def load_pcd_open3d(filepath: str) -> np.ndarray:
    """Open3D .pcd/.ply → (N,4) float32 (intensity column = 0)."""
    import open3d as o3d
    pcd = o3d.io.read_point_cloud(filepath)
    xyz = np.asarray(pcd.points, dtype=np.float32)
    return np.hstack([xyz, np.zeros((len(xyz), 1), dtype=np.float32)])


def ingest_point_cloud(src: str, dst_npy: str) -> None:
    """Convert .bin / .pcd / .ply to .npy."""
    ext = Path(src).suffix.lower()
    if   ext == ".bin":            pts = load_bin(src)
    elif ext in (".pcd", ".ply"): pts = load_pcd_open3d(src)
    elif ext == ".npy":            pts = np.load(src).astype(np.float32)
    else: raise ValueError(f"Unsupported format: {ext}")
    np.save(dst_npy, pts)


def batch_ingest(src_dir: str, dst_dir: Path,
                 exts=(".bin", ".pcd", ".ply", ".npy")) -> list:
    src_dir = Path(src_dir)
    ids = []
    for f in sorted(f for f in src_dir.rglob("*") if f.suffix.lower() in exts):
        ingest_point_cloud(str(f), str(dst_dir / f"{f.stem}.npy"))
        ids.append(f.stem)
    log.info("Ingested %d point clouds.", len(ids))
    return ids


# ── Label helpers ─────────────────────────────────────────────────────────────

def write_label_file(save_path: Path, boxes: np.ndarray, names: list) -> None:
    """Write KITTI-style label: cx cy cz l w h yaw ClassName per line."""
    with open(save_path, "w") as f:
        for box, name in zip(boxes, names):
            f.write(f"{box[0]:.4f} {box[1]:.4f} {box[2]:.4f} "
                    f"{box[3]:.4f} {box[4]:.4f} {box[5]:.4f} "
                    f"{box[6]:.4f} {name}\n")


def parse_label_file(p: Path):
    boxes, names = [], []
    with open(p) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8: continue
            boxes.append([float(x) for x in parts[:7]])
            names.append(parts[7])
    return np.array(boxes, dtype=np.float32), names


# ── Generate synthetic scenes ─────────────────────────────────────────────────

rng = np.random.default_rng(42)
N_SCENES = 60    # ← increase when you have real data; aim for 500–2000+

# Synthetic object templates  [l, w, h]
OBJ_TEMPLATES = {
    "SilverPlate": np.array([0.60, 0.60, 0.05]),
    "SafetyCone":  np.array([0.45, 0.45, 1.00]),
    "Barricade":   np.array([2.00, 0.50, 1.00]),
}

sample_ids = []
for i in range(N_SCENES):
    sid = f"{i:06d}"
    sample_ids.append(sid)

    # Background: sparse outdoor ground plane
    n_bg = rng.integers(800, 2000)
    bg_xy  = rng.uniform(-40, 40, (n_bg, 2)).astype(np.float32)
    bg_z   = rng.uniform(-0.5, 0.1, (n_bg, 1)).astype(np.float32)
    bg_int = rng.uniform(0.0, 0.3, (n_bg, 1)).astype(np.float32)
    pts = np.hstack([bg_xy, bg_z, bg_int])

    boxes, names = [], []
    n_objs = rng.integers(1, 5)
    for _ in range(n_objs):
        cls  = rng.choice(CLASS_NAMES)
        lwh  = OBJ_TEMPLATES[cls] * rng.uniform(0.9, 1.1, 3)
        cx   = float(rng.uniform(-20, 20))
        cy   = float(rng.uniform(-20, 20))
        cz   = float(lwh[2] / 2 - 0.5)
        yaw  = float(rng.uniform(-np.pi, np.pi))

        # Simulate object points
        n_obj_pts = rng.integers(8, 60)
        ox = rng.uniform(-lwh[0]/2, lwh[0]/2, n_obj_pts).astype(np.float32)
        oy = rng.uniform(-lwh[1]/2, lwh[1]/2, n_obj_pts).astype(np.float32)
        oz = rng.uniform(0, lwh[2], n_obj_pts).astype(np.float32)
        # Rotate & translate
        cos_y, sin_y = np.cos(yaw), np.sin(yaw)
        rx = cos_y * ox - sin_y * oy + cx
        ry = sin_y * ox + cos_y * oy + cy
        rz = oz + cz - lwh[2]/2
        # High intensity for SilverPlate (reflective)
        inten = (rng.uniform(0.8, 1.0, n_obj_pts) if cls == "SilverPlate"
                 else rng.uniform(0.3, 0.7, n_obj_pts)).astype(np.float32)
        obj_pts = np.stack([rx, ry, rz, inten], axis=1)
        pts = np.vstack([pts, obj_pts])

        boxes.append([cx, cy, cz, float(lwh[0]), float(lwh[1]), float(lwh[2]), yaw])
        names.append(cls)

    np.save(str(DATA_ROOT / "points" / f"{sid}.npy"), pts.astype(np.float32))
    write_label_file(DATA_ROOT / "labels" / f"{sid}.txt",
                     np.array(boxes, dtype=np.float32), names)

print(f"✅ Generated {N_SCENES} synthetic scenes → {DATA_ROOT}")

---
## Step 5 — Train / Val / Test Split

In [ ]:
def generate_splits(ids, root, train_ratio=0.75, val_ratio=0.15, seed=42):
    rng2 = random.Random(seed)
    ids  = list(ids)
    rng2.shuffle(ids)
    n  = len(ids)
    nt = int(n * train_ratio)
    nv = int(n * val_ratio)
    splits = {
        "train": ids[:nt],
        "val":   ids[nt: nt+nv],
        "test":  ids[nt+nv:],
    }
    for split, sids in splits.items():
        path = root / "ImageSets" / f"{split}.txt"
        with open(path, "w") as f:
            f.write("\n".join(sids) + "\n")
        log.info("Split %-5s : %d samples", split, len(sids))
    return splits

splits = generate_splits(sample_ids, DATA_ROOT)
print("✅ Splits written to ImageSets/")

---
## Step 6 — Dataset Statistics

In [ ]:
from collections import Counter

def dataset_statistics(root: Path) -> dict:
    counts = Counter()
    label_files = sorted((root / "labels").glob("*.txt"))
    for lf in label_files:
        _, names = parse_label_file(lf)
        counts.update(names)

    print("\n📊 Dataset Statistics")
    print("-" * 35)
    print(f"  Total scenes : {len(label_files)}")
    print(f"  Total objects: {sum(counts.values())}")
    print()
    for cls in CLASS_NAMES:
        n = counts.get(cls, 0)
        bar = "█" * (n // max(1, sum(counts.values()) // 30))
        print(f"  {cls:<15} {n:>5}   {bar}")
    print()

    if any(counts.get(c, 0) < 50 for c in CLASS_NAMES):
        print("⚠️  Some classes have < 50 instances. "
              "Aim for 500–2000+ labeled frames for good finetuning.")
    else:
        print("✅ Instance counts look reasonable.")
    return dict(counts)

stats = dataset_statistics(DATA_ROOT)

---
## Step 7 — Config Generation
Generates two YAML files directly inside the OpenPCDet tree:
- `tools/cfgs/dataset_configs/excavation_dataset.yaml`
- `tools/cfgs/custom_models/excavation_pointpillars.yaml`

In [ ]:
# ── 7a — Dataset YAML ─────────────────────────────────────────────────────────

def make_dataset_yaml(root: Path, output_path: Path) -> None:
    sample_groups  = [f"{c}:20" for c in CLASS_NAMES]
    filter_min_pts = [f"{c}:3"  for c in CLASS_NAMES]

    cfg = {
        "DATASET":   "CustomDataset",
        "DATA_PATH": str(root),
        "POINT_CLOUD_RANGE": POINT_CLOUD_RANGE,
        "MAP_CLASS_TO_KITTI": MAP_CLASS_TO_KITTI,
        "DATA_SPLIT": {"train": "train", "test": "val"},
        "INFO_PATH":  {
            "train": ["custom_infos_train.pkl"],
            "test":  ["custom_infos_val.pkl"],
        },
        "POINT_FEATURE_ENCODING": {
            "encoding_type":   "absolute_coordinates_encoding",
            "used_feature_list": ["x", "y", "z", "intensity"],
            "src_feature_list":  ["x", "y", "z", "intensity"],
        },
        "DATA_AUGMENTOR": {
            "DISABLE_AUG_LIST": ["placeholder"],
            "AUG_CONFIG_LIST": [
                {
                    "NAME": "gt_sampling",
                    "USE_ROAD_PLANE": False,
                    "DB_INFO_PATH": ["custom_dbinfos_train.pkl"],
                    "PREPARE": {"filter_by_min_points": filter_min_pts},
                    "SAMPLE_GROUPS": sample_groups,
                    "NUM_POINT_FEATURES": 4,
                    "DATABASE_WITH_FAKELIDAR": False,
                    "REMOVE_EXTRA_WIDTH": [0.0, 0.0, 0.0],
                    "LIMIT_WHOLE_SCENE": True,
                },
                {"NAME": "random_world_flip",     "ALONG_AXIS_LIST": ["x"]},
                {"NAME": "random_world_rotation", "WORLD_ROT_ANGLE": [-0.3927, 0.3927]},
                {"NAME": "random_world_scaling",  "WORLD_SCALE_RANGE": [0.95, 1.05]},
            ],
        },
        "DATA_PROCESSOR": [
            {"NAME": "mask_points_and_boxes_outside_range", "REMOVE_OUTSIDE_BOXES": True},
            {"NAME": "shuffle_points", "SHUFFLE_ENABLED": {"train": True, "test": False}},
            {
                "NAME": "transform_points_to_voxels",
                "VOXEL_SIZE": VOXEL_SIZE,
                "MAX_POINTS_PER_VOXEL": 5,
                "MAX_NUMBER_OF_VOXELS": {"train": 150000, "test": 150000},
            },
        ],
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    log.info("Dataset YAML → %s", output_path)

make_dataset_yaml(DATA_ROOT, DATASET_YAML)
print("✅ Dataset YAML written.")

In [ ]:
# ── 7b — Model YAML (PointPillars) ────────────────────────────────────────────

def _anchor_entry(cls_name: str, stride: int = 8) -> dict:
    sizes = ANCHOR_SIZES[cls_name]
    matched, unmatched = IOU_THRESHOLDS[cls_name]
    return {
        "class_name":          cls_name,
        "anchor_sizes":        [sizes],
        "anchor_rotations":    [0, 1.5708],
        "anchor_bottom_heights": [0],
        "align_center":        False,
        "feature_map_stride":  stride,
        "matched_threshold":   matched,
        "unmatched_threshold": unmatched,
    }


def make_model_yaml(dataset_yaml_path: Path, output_path: Path) -> None:
    cfg = {
        "CLASS_NAMES": CLASS_NAMES,
        "DATA_CONFIG": {"_BASE_CONFIG_": str(dataset_yaml_path)},
        "MODEL": {
            "NAME": "PointPillar",
            "VFE": {
                "NAME": "PillarVFE",
                "WITH_DISTANCE": False,
                "USE_ABSLOTE_XYZ": True,
                "USE_NORM": True,
                "NUM_FILTERS": [64],
            },
            "MAP_TO_BEV": {
                "NAME": "PointPillarScatter",
                "NUM_BEV_FEATURES": 64,
            },
            "BACKBONE_2D": {
                "NAME": "BaseBEVBackbone",
                "LAYER_NUMS":            [3, 5, 5],
                "LAYER_STRIDES":         [2, 2, 2],
                "NUM_FILTERS":           [64, 128, 256],
                "UPSAMPLE_STRIDES":      [1, 2, 4],
                "NUM_UPSAMPLE_FILTERS": [128, 128, 128],
            },
            "DENSE_HEAD": {
                "NAME": "AnchorHeadSingle",
                "CLASS_AGNOSTIC": False,
                "USE_DIRECTION_CLASSIFIER": True,
                "DIR_OFFSET": 0.78539,
                "DIR_LIMIT_OFFSET": 0.0,
                "NUM_DIR_BINS": 2,
                "ANCHOR_GENERATOR_CONFIG": [_anchor_entry(c) for c in CLASS_NAMES],
                "TARGET_ASSIGNER_CONFIG": {
                    "NAME": "AxisAlignedTargetAssigner",
                    "POS_FRACTION": -1.0,
                    "SAMPLE_SIZE": 512,
                    "NORM_BY_NUM_EXAMPLES": False,
                    "MATCH_HEIGHT": False,
                    "BOX_CODER": "ResidualCoder",
                },
                "LOSS_CONFIG": {
                    "LOSS_WEIGHTS": {
                        "cls_weight":   1.0,
                        "loc_weight":   2.0,
                        "dir_weight":   0.2,
                        "code_weights": [1.0] * 7,
                    }
                },
            },
            "POST_PROCESSING": {
                "RECALL_THRESH_LIST": [0.3, 0.5, 0.7],
                "SCORE_THRESH": 0.1,
                "OUTPUT_RAW_SCORE": False,
                "EVAL_METRIC": "kitti",
                "NMS_CONFIG": {
                    "MULTI_CLASSES_NMS": False,
                    "NMS_TYPE": "nms_gpu",
                    "NMS_THRESH": 0.5,
                    "NMS_PRE_MAXSIZE": 4096,
                    "NMS_POST_MAXSIZE": 500,
                },
            },
        },
        "OPTIMIZATION": {
            "BATCH_SIZE_PER_GPU": 4,
            "NUM_EPOCHS": 80,
            "OPTIMIZER": "adam_onecycle",
            "LR": 0.001,
            "WEIGHT_DECAY": 0.01,
            "MOMENTUM": 0.9,
            "MOMS": [0.95, 0.85],
            "PCT_START": 0.4,
            "DIV_FACTOR": 10,
            "DECAY_STEP_LIST": [35, 45],
            "LR_DECAY": 0.1,
            "LR_CLIP": 1e-7,
            "LR_WARMUP": True,
            "WARMUP_EPOCH": 2,
            "GRAD_NORM_CLIP": 10,
        },
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    log.info("Model YAML → %s", output_path)

make_model_yaml(DATASET_YAML, MODEL_YAML)
print("✅ Model YAML written.")

In [ ]:
# Preview the dataset YAML
print("=" * 60)
print("  excavation_dataset.yaml  (first 50 lines)")
print("=" * 60)
with open(DATASET_YAML) as f:
    lines = f.readlines()
print("".join(lines[:50]))

---
## Step 8 — Info Files & GT Database
Calls OpenPCDet's `create_custom_infos()` to:
1. Build `custom_infos_train.pkl` and `custom_infos_val.pkl`
2. Extract per-object point clouds into `gt_database/` for GT-sampling augmentation

In [ ]:
import importlib, sys

# Make sure OpenPCDet is importable
if str(OPENPCDET_ROOT) not in sys.path:
    sys.path.insert(0, str(OPENPCDET_ROOT))

from easydict import EasyDict
from pcdet.datasets.custom.custom_dataset import create_custom_infos
from pcdet.utils import common_utils

dataset_cfg = EasyDict(yaml.safe_load(open(DATASET_YAML)))

# Override DATA_PATH to absolute path
dataset_cfg.DATA_PATH = str(DATA_ROOT)

create_custom_infos(
    dataset_cfg=dataset_cfg,
    class_names=CLASS_NAMES,
    data_path=DATA_ROOT,
    save_path=DATA_ROOT,
    workers=2,
)
print("\n✅ Info files + GT database created.")

In [ ]:
# Quick sanity check on the generated pickle
with open(DATA_ROOT / "custom_infos_train.pkl", "rb") as f:
    infos = pickle.load(f)

print(f"Train info entries : {len(infos)}")
print(f"Sample keys        : {list(infos[0].keys())}")
if "annos" in infos[0]:
    annos = infos[0]["annos"]
    print(f"Classes in frame 0 : {annos['name'].tolist()}")
    print(f"GT boxes shape     : {annos['gt_boxes_lidar'].shape}")

---
## Step 9 — Semi-Automatic Labelling Helper
DBSCAN clustering + Open3D oriented bounding boxes to bootstrap annotations on unlabelled scans.

In [ ]:
def semi_automatic_label(points: np.ndarray,
                          eps: float = 0.4,
                          min_cluster_pts: int = 5) -> list:
    """
    DBSCAN cluster → oriented bounding box → heuristic class assignment.
    Returns list of dicts: {class_name, box_3d=[x,y,z,l,w,h,yaw], num_points}
    """
    import open3d as o3d
    from sklearn.cluster import DBSCAN

    xyz    = points[:, :3]
    labels = DBSCAN(eps=eps, min_samples=min_cluster_pts).fit_predict(xyz)

    results = []
    for lid in set(labels):
        if lid < 0: continue        # noise
        mask = labels == lid
        cluster = xyz[mask]

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(cluster)
        obb    = pcd.get_oriented_bounding_box()
        center = np.array(obb.center)
        extent = np.sort(np.array(obb.extent))  # [min → max]
        l, w, h = float(extent[2]), float(extent[1]), float(extent[0])
        R   = np.array(obb.R)
        yaw = float(np.arctan2(R[1, 0], R[0, 0]))

        # Heuristic: use geometry to guess class
        if   h < 0.15 and l < 1.0: cls = "SilverPlate"
        elif h > 0.60 and l < 0.7: cls = "SafetyCone"
        else:                       cls = "Barricade"

        results.append({
            "class_name": cls,
            "box_3d": [float(center[0]), float(center[1]), float(center[2]),
                       l, w, h, yaw],
            "num_points": int(mask.sum()),
        })

    return results


# ── Demo on one synthetic frame ───────────────────────────────────────────────
demo_pts = np.load(str(DATA_ROOT / "points" / f"{sample_ids[0]}.npy"))
proposals = semi_automatic_label(demo_pts, eps=0.6, min_cluster_pts=4)

print(f"Semi-auto found {len(proposals)} candidate objects:")
for p in proposals:
    b = p['box_3d']
    print(f"  {p['class_name']:<15}  "
          f"center=({b[0]:.2f},{b[1]:.2f},{b[2]:.2f})  "
          f"lwh=({b[3]:.2f},{b[4]:.2f},{b[5]:.2f})  "
          f"pts={p['num_points']}")

---
## Step 10 — BEV Visualisation of a Scene
Quick matplotlib Bird's-Eye-View sanity check before training.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CLASS_COLORS = {
    "SilverPlate": "gold",
    "SafetyCone":  "orangered",
    "Barricade":   "dodgerblue",
}

def draw_bev(pts, boxes, names, title="Bird's-Eye View"):
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.set_facecolor("#1a1a2e")
    fig.patch.set_facecolor("#0f0f1a")

    # Points coloured by intensity
    inten = pts[:, 3] if pts.shape[1] >= 4 else np.zeros(len(pts))
    ax.scatter(pts[:, 0], pts[:, 1], c=inten, cmap="plasma",
               s=0.4, alpha=0.6, vmin=0, vmax=1)

    # GT boxes
    for box, name in zip(boxes, names):
        cx, cy, _, l, w, _, yaw = box
        color = CLASS_COLORS.get(name, "white")

        corners_local = np.array([
            [ l/2,  w/2],
            [ l/2, -w/2],
            [-l/2, -w/2],
            [-l/2,  w/2],
        ])
        cos_y, sin_y = np.cos(yaw), np.sin(yaw)
        R2 = np.array([[cos_y, -sin_y], [sin_y, cos_y]])
        corners = (R2 @ corners_local.T).T + [cx, cy]

        poly = plt.Polygon(corners, fill=False, edgecolor=color, linewidth=1.5)
        ax.add_patch(poly)
        ax.text(cx, cy, name, color=color, fontsize=5,
                ha="center", va="center", fontweight="bold")

    ax.set_xlim(-45, 45); ax.set_ylim(-45, 45)
    ax.set_xlabel("X (m)", color="white"); ax.set_ylabel("Y (m)", color="white")
    ax.tick_params(colors="white")
    ax.set_title(title, color="white", fontsize=13)
    legend_handles = [patches.Patch(color=c, label=n)
                      for n, c in CLASS_COLORS.items()]
    ax.legend(handles=legend_handles, loc="upper right",
              facecolor="#1a1a2e", labelcolor="white")
    plt.tight_layout()
    plt.savefig("/kaggle/working/bev_sample.png", dpi=150, bbox_inches="tight")
    plt.show()


# Visualise first training scene
demo_sid   = splits["train"][0]
demo_pts   = np.load(str(DATA_ROOT / "points" / f"{demo_sid}.npy"))
demo_boxes, demo_names = parse_label_file(DATA_ROOT / "labels" / f"{demo_sid}.txt")
draw_bev(demo_pts, demo_boxes, demo_names,
         title=f"BEV — scene {demo_sid}  ({len(demo_names)} objects)")
print("✅ BEV saved to /kaggle/working/bev_sample.png")

---
## Step 11 — Training

> 💡 **Tips:**
> - Provide a pretrained checkpoint via `PRETRAINED_CKPT` (e.g. nuScenes PointPillars) for fast convergence.
> - For a quick smoke-test set `QUICK_SMOKE_TEST = True` (trains 2 epochs).
> - Kaggle gives you 9–12 h of GPU; 80 epochs on ~60 scenes finishes in minutes.

In [ ]:
# ── Training configuration ─────────────────────────────────────────────────────
PRETRAINED_CKPT = None   # e.g. "/kaggle/input/my-pretrained/nuscenes_pp.pth"
QUICK_SMOKE_TEST = True  # Set False for a full training run
NUM_EPOCHS  = 2 if QUICK_SMOKE_TEST else 80
BATCH_SIZE  = 2          # Reduce if OOM; increase for faster training
EXTRA_TAG   = "excavation_finetune"

print(f"Epochs     : {NUM_EPOCHS}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Smoke test : {QUICK_SMOKE_TEST}")

In [ ]:
import subprocess, os

def run_subprocess(cmd, cwd=None):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(OPENPCDET_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    print("\n▶ " + " ".join(str(c) for c in cmd) + "\n")
    result = subprocess.run(
        [str(c) for c in cmd],
        env=env, cwd=str(cwd) if cwd else None,
        check=True,
    )
    return result


train_cmd = [
    sys.executable,
    OPENPCDET_ROOT / "tools" / "train.py",
    "--cfg_file",  MODEL_YAML,
    "--batch_size", str(BATCH_SIZE),
    "--epochs",     str(NUM_EPOCHS),
    "--output_dir", OUTPUT_DIR,
    "--extra_tag",  EXTRA_TAG,
    "--workers",    "0",       # Kaggle containers: avoid multiprocessing issues
]

if PRETRAINED_CKPT and Path(PRETRAINED_CKPT).exists():
    train_cmd += ["--pretrained_model", PRETRAINED_CKPT]

run_subprocess(train_cmd, cwd=OPENPCDET_ROOT / "tools")
print("\n✅ Training complete.")

---
## Step 12 — Evaluation (mAP)
Runs KITTI-style evaluation on the val split and prints mAP @ IoU 0.5 / 0.7.

In [ ]:
# Auto-find the latest checkpoint
ckpts = sorted(OUTPUT_DIR.rglob("*.pth"))
if not ckpts:
    print("⚠️  No checkpoint found. Run the training cell first.")
else:
    latest_ckpt = ckpts[-1]
    print(f"Using checkpoint: {latest_ckpt}")

    eval_cmd = [
        sys.executable,
        OPENPCDET_ROOT / "tools" / "test.py",
        "--cfg_file",   MODEL_YAML,
        "--batch_size", str(BATCH_SIZE),
        "--ckpt",       latest_ckpt,
        "--output_dir", OUTPUT_DIR / "eval",
        "--extra_tag",  "eval",
        "--workers",    "0",
    ]

    run_subprocess(eval_cmd, cwd=OPENPCDET_ROOT / "tools")
    print("\n✅ Evaluation complete. Check the output above for mAP scores.")

---
## Step 13 — Inference on a New Point Cloud
Runs the trained model on a single scan and saves a BEV result image.

In [ ]:
# ── Pick any .npy from the test split (or supply your own .bin/.pcd/.ply) ──────
test_ids = splits.get("test", splits["val"])
if not test_ids:
    test_ids = [sample_ids[-1]]

INFER_SCENE = test_ids[0]
infer_npy   = DATA_ROOT / "points" / f"{INFER_SCENE}.npy"

# Convert .npy → .bin (demo.py expects .bin by default)
infer_bin = Path("/kaggle/working/infer_demo.bin")
pts_infer = np.load(str(infer_npy)).astype(np.float32)
pts_infer.tofile(str(infer_bin))
print(f"Inference on scene: {INFER_SCENE}  |  {len(pts_infer)} points")

ckpts = sorted(OUTPUT_DIR.rglob("*.pth"))
if not ckpts:
    print("⚠️  No checkpoint found. Run the training cell first.")
else:
    latest_ckpt = ckpts[-1]
    infer_out   = OUTPUT_DIR / "inference"
    infer_out.mkdir(parents=True, exist_ok=True)

    infer_cmd = [
        sys.executable,
        OPENPCDET_ROOT / "tools" / "demo.py",
        "--cfg_file",   MODEL_YAML,
        "--ckpt",       latest_ckpt,
        "--data_path",  infer_bin,
        "--ext",        ".bin",
        "--output_dir", infer_out,
    ]

    run_subprocess(infer_cmd, cwd=OPENPCDET_ROOT / "tools")
    print("\n✅ Inference complete. Results saved to:", infer_out)

---
## Step 14 — BEV Visualisation of Inference Results

In [ ]:
# Load GT boxes for the inference scene and display alongside semi-auto proposals
gt_boxes_infer, gt_names_infer = parse_label_file(
    DATA_ROOT / "labels" / f"{INFER_SCENE}.txt"
)

proposals_infer = semi_automatic_label(pts_infer, eps=0.6, min_cluster_pts=4)
prop_boxes = np.array([p["box_3d"] for p in proposals_infer], dtype=np.float32) \
    if proposals_infer else np.zeros((0, 7), dtype=np.float32)
prop_names = [p["class_name"] for p in proposals_infer]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor("#0f0f1a")

for ax, boxes, names, title in [
    (axes[0], gt_boxes_infer, gt_names_infer, "Ground-Truth Boxes"),
    (axes[1], prop_boxes,     prop_names,     "Semi-Auto Proposals (DBSCAN)"),
]:
    ax.set_facecolor("#1a1a2e")
    ax.scatter(pts_infer[:, 0], pts_infer[:, 1],
               c=pts_infer[:, 3], cmap="plasma", s=0.3, alpha=0.6, vmin=0, vmax=1)
    for box, name in zip(boxes, names):
        cx, cy, _, l, w, _, yaw = box
        color = CLASS_COLORS.get(name, "white")
        corners_l = np.array([[ l/2, w/2],[ l/2,-w/2],[-l/2,-w/2],[-l/2, w/2]])
        cos_y, sin_y = np.cos(yaw), np.sin(yaw)
        R2 = np.array([[cos_y,-sin_y],[sin_y,cos_y]])
        corners = (R2 @ corners_l.T).T + [cx, cy]
        ax.add_patch(plt.Polygon(corners, fill=False, edgecolor=color, linewidth=1.8))
        ax.text(cx, cy, name[:4], color=color, fontsize=5, ha="center", va="center")
    ax.set_xlim(-45,45); ax.set_ylim(-45,45)
    ax.set_title(title, color="white", fontsize=11)
    ax.tick_params(colors="white")
    ax.set_xlabel("X (m)", color="white")
    ax.set_ylabel("Y (m)", color="white")

legend_handles = [patches.Patch(color=c, label=n) for n, c in CLASS_COLORS.items()]
axes[1].legend(handles=legend_handles, loc="upper right",
               facecolor="#1a1a2e", labelcolor="white")
plt.tight_layout()
plt.savefig("/kaggle/working/bev_inference.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → /kaggle/working/bev_inference.png")

---
## Summary & Next Steps

| Done | Task |
|------|------|
| ✅ | OpenPCDet installed + compiled |
| ✅ | Dataset layout created |
| ✅ | Point clouds ingested (`.bin`/`.pcd`/`.ply` → `.npy`) |
| ✅ | KITTI-style label files written |
| ✅ | Train/val/test split generated |
| ✅ | Dataset + model YAML configs generated |
| ✅ | Info files + GT database created |
| ✅ | Model trained (PointPillars) |
| ✅ | Evaluation (mAP) run |
| ✅ | Inference + BEV visualisation |

### To improve mAP

1. **More data** — aim for **500–2000+ labeled frames** per class; use real LiDAR scans.
2. **Better pretrained weights** — download nuScenes or KITTI PointPillars from the [OpenPCDet model zoo](https://github.com/open-mmlab/OpenPCDet/blob/master/docs/MODEL_ZOO.md) and pass to `--pretrained_model`.
3. **Intensity features** — SilverPlate is highly reflective; intensity is already enabled.
4. **Smaller voxels** — already set to `[0.05, 0.05, 0.10]` m for fine-grained detail.
5. **Swap backbone** — try **CenterPoint** (heatmap-based) for better small-object recall:
   ```bash
   cp tools/cfgs/nuscenes_models/cbgs_voxel01_res3d_centerpoint.yaml \
      tools/cfgs/custom_models/excavation_centerpoint.yaml
   # Then update CLASS_NAMES, anchor sizes, and DATA_CONFIG._BASE_CONFIG_
   ```
6. **Manual review** — use the semi-auto proposals (Step 9) in LabelCloud or Supervisely to correct labels faster.